<a href="https://colab.research.google.com/github/justorfc/Estadistica_Aplicada_con_Python_y_R/blob/main/12_Semana_12_Validaci%C3%B3n_de_Modelos%2C_M%C3%A9tricas_y_Ciencia_Reproducible.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Este Notebook es la propuesta estructurada para la **Semana 12**. Con esta semana cerramos el **Eje IV** y marcamos un hito crucial: pasamos de simplemente "ajustar" modelos a **validarlos rigurosamente**, introduciendo el flujo de trabajo de Machine Learning (partición de datos) y consolidando las prácticas de ciencia reproducible (GitHub y Quarto).

# Semana 12: Validación de Modelos, Métricas y Ciencia Reproducible

**Resultado de aprendizaje:** Audita, valida y comunica modelos estadísticos de forma reproducible, distinguiendo el ajuste de entrenamiento de la capacidad de generalización, utilizando control de versiones y el modo agente (human-in-the-loop).

---

#### Sesión 1: Partición de Datos y Métricas de Error (80 - 90 minutos)

**Objetivo:** Comprender por qué evaluar un modelo con los mismos datos que usó para aprender es un error metodológico grave (sobreajuste), y utilizar `scikit-learn` en Python para particionar y evaluar métricas reales.

* **20 min - Diálogo socrático y conceptualización (Lápiz y papel):**
* *Situación:* Si le das a un estudiante el examen final resuelto antes de la prueba, sacará 100%. ¿Pero realmente aprendió la materia o solo memorizó las respuestas? Lo mismo ocurre con los modelos matemáticos.
* *Actividad:* Explicación del flujo de partición de datos: Conjunto de Entrenamiento (Train) para aprender, y Conjunto de Prueba (Test) para validar. Introducción a las métricas de error absolutas y cuadráticas (MAE, RMSE).


* **45 min - Exploración en Google Colab (Python):**
* Carga del cuaderno de la semana 12.
* Introducción a `scikit-learn` (la librería estándar de Machine Learning en Python).
* Uso de `train_test_split` para separar los datos (80% / 20%).
* Cálculo e interpretación de métricas: MAE (Error Absoluto Medio) y RMSE (Raíz del Error Cuadrático Medio).


* **15 min - Reflexión manuscrita:**
* Análisis de los resultados: ¿Por qué el error en el conjunto de prueba suele ser mayor que en el de entrenamiento? ¿Qué significa esto para la incertidumbre agrícola?



---

#### Sesión 2: Ciencia Reproducible, GitHub y Quarto (80 - 90 minutos)

**Objetivo:** Integrar la validación estadística con las mejores prácticas de la ingeniería de software: documentación, trazabilidad y reportes reproducibles.

* **25 min - El ecosistema de la Ciencia Reproducible:**
* Explicación de Git y GitHub: El concepto de "control de versiones" como una máquina del tiempo para el código.
* Transición de RMarkdown a **Quarto** (la evolución moderna y multilingüe para informes científicos y técnicos).


* **25 min - Prompts para validación y métricas en R:**
* Demostración de cómo instruir a la IA para replicar la partición de datos en R usando el paquete `rsample` o `initial_split` de `tidymodels`, y cómo calcular MAE/RMSE.


* **40 min - Reto en Posit Cloud (Auditoría final del Eje IV):**
* Los estudiantes ejecutan el flujo de entrenamiento y prueba en un documento Quarto (`.qmd`), auditan los cambios sugeridos por su agente de IA, compilan el reporte y configuran su bitácora de IA evidenciando el control humano (human-in-the-loop).



---

A continuación, el contenido listo para integrarse en las celdas de tu cuaderno de Google Colab.

---

### Celda de Texto 1

```markdown
# Semana 12: Validación Predictiva y Métricas de Error
**Asignatura:** Estadística Aplicada con Python y R  
**Programa:** Ingeniería Agrícola - Universidad de Sucre  
**Profesor:** Justo Rafael Fuentes Cuello  

---

### Situación de Interés: La Trampa de la Memorización
Hasta ahora, hemos medido el éxito de nuestros modelos viendo cómo se ajustan a los datos que usamos para crearlos (usando el $R^2$). Sin embargo, en el mundo real, un modelo no sirve para explicar el pasado, sirve para **predecir el futuro con datos nuevos**.

Si evaluamos a un modelo con sus propios datos de entrenamiento, corremos el riesgo de **Sobreajuste (Overfitting)**: el modelo memoriza el ruido en lugar de aprender el patrón. Hoy introducimos el rigor del Machine Learning: dividiremos nuestros datos en un conjunto de "Entrenamiento" (para enseñar al modelo) y un conjunto "Ciego de Prueba" (para ver si realmente sabe predecir).

```

### Celda de Código 1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Importamos la librería líder de Machine Learning en Python: scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="ticks")
np.random.seed(12)

# Simulamos 200 registros históricos de Evapotranspiración (ETo) diaria
# Variables independientes: Temperatura Máxima, Radiación Solar y Velocidad del Viento
temp_max = np.random.uniform(28, 38, 200)
radiacion = np.random.uniform(10, 25, 200)
viento = np.random.uniform(1, 6, 200)

# ETo (mm/día) = Fórmula base + Efectos + Ruido
ruido = np.random.normal(0, 0.5, 200)
eto_mm = 1.2 + (0.05 * temp_max) + (0.15 * radiacion) + (0.2 * viento) + ruido

df_clima = pd.DataFrame({
    'Temp_Max_C': temp_max,
    'Radiacion_MJ': radiacion,
    'Viento_ms': viento,
    'ETo_mm': eto_mm
})

print("Dataset de clima simulado generado. 200 días de registros.")
df_clima.head()

### 1. Partición de los Datos (Train / Test Split)
Vamos a esconderle al modelo el 20% de los datos.
Solo usaremos el 80% (Entrenamiento) para calcular los coeficientes matemáticos. Luego, le pediremos que prediga los resultados del 20% restante (Prueba) y compararemos sus predicciones con la realidad.

```

### Celda de Código 2

In [ ]:
# 1. Separamos nuestras variables predictoras (X) y nuestra variable objetivo (y)
X = df_clima[['Temp_Max_C', 'Radiacion_MJ', 'Viento_ms']]
y = df_clima['ETo_mm']

# 2. Hacemos la partición (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Total de datos: {len(df_clima)}")
print(f"Datos para ENTRENAR al modelo (Train): {len(X_train)}")
print(f"Datos para EXAMINAR al modelo (Test): {len(X_test)} (¡El modelo nunca verá estos datos hasta el final!)")

### 2. Entrenamiento del Modelo de Machine Learning
Esta vez no usaremos `statsmodels`, usaremos la clase `LinearRegression` de `scikit-learn`, que es el estándar de la industria para construir flujos de trabajo predictivos.
Solo alimentaremos el modelo con `X_train` y `y_train`.

```

### Celda de Código 3

In [ ]:
# Inicializamos el modelo de regresión
modelo_ml = LinearRegression()

# ENTRENAMIENTO: Ajustamos el modelo solo con el 80% de los datos
modelo_ml.fit(X_train, y_train)

# Mostramos los coeficientes aprendidos
print("Modelo entrenado exitosamente.")
print(f"Intercepto: {modelo_ml.intercept_:.3f}")
for col, coef in zip(X.columns, modelo_ml.coef_):
    print(f"Coeficiente para {col}: {coef:.4f}")

### 3. El Momento de la Verdad: Predicción y Métricas
Ahora tomamos nuestro modelo entrenado y le pasamos los datos del clima de los 40 días que guardamos para prueba (`X_test`). El modelo nos dará sus predicciones.

Evaluaremos el error real usando dos métricas vitales en ingeniería:
1.  **MAE (Error Absoluto Medio):** En promedio, ¿cuántos milímetros de ETo nos estamos equivocando?
2.  **RMSE (Raíz del Error Cuadrático Medio):** Similar al MAE, pero penaliza severamente los errores grandes. Se calcula como $\sqrt{\frac{1}{n}\sum(y_{real} - y_{pred})^2}$.

```

### Celda de Código 4

In [ ]:
# Hacemos que el modelo prediga sobre el conjunto de prueba
predicciones_test = modelo_ml.predict(X_test)

# Calculamos las métricas
mae = mean_absolute_error(y_test, predicciones_test)
rmse = np.sqrt(mean_squared_error(y_test, predicciones_test)) # RMSE es la raíz cuadrada del MSE
r2_test = r2_score(y_test, predicciones_test)

print("--- RESULTADOS DE LA VALIDACIÓN (CONJUNTO DE PRUEBA) ---")
print(f"MAE  (Error Absoluto Medio): {mae:.3f} mm/día")
print(f"RMSE (Raíz Error Cuadrático): {rmse:.3f} mm/día")
print(f"R²   (Poder Predictivo): {r2_test*100:.1f}%")

# Gráfico de Realidad vs Predicción
plt.figure(figsize=(6, 6))
plt.scatter(y_test, predicciones_test, color='darkcyan', alpha=0.7)
# Línea de perfección (Realidad = Predicción)
limites = [y_test.min(), y_test.max()]
plt.plot(limites, limites, 'r--', lw=2, label='Predicción Perfecta')

plt.title('Evaluación de Generalización: ETo Real vs Predicha')
plt.xlabel('ETo Real (mm/día) - Datos de Prueba')
plt.ylabel('ETo Predicha por el Modelo (mm/día)')
plt.legend()
plt.show()

### 🛑 Reflexión y Reserva Cognitiva (Síntesis manuscrita)
Toma tu lápiz y tu bitácora para registrar tus conclusiones de validación:
1. El **MAE** nos dio un valor en "mm/día". Como ingeniero diseñando un reservorio de agua agrícola, ¿consideras que equivocarte por esa cantidad diaria es un error aceptable o crítico para el cultivo?
2. Revisa el gráfico "Real vs Predicha". Si todos los puntos cayeran exactamente sobre la línea roja punteada, ¿qué significaría? ¿Por qué en los sistemas físicos reales nunca obtenemos esa línea perfecta?
3. Imagina que un modelo de IA te da un R² del 99% en el conjunto de Entrenamiento, pero cuando lo evalúas en el conjunto de Prueba, el R² cae al 40%. Escribe con tus palabras qué fenómeno ocurrió y por qué ese modelo es un peligro para la ingeniería.

---

### Instrucciones para el reto en R (Trabajo Autónomo y Sesión 2)

**Misión:** La transición al Machine Learning requiere rigor. Ahora deberás aplicar el concepto de `train_test_split` y cálculo de métricas en **R**, pero esta vez documentando todo bajo el estándar moderno de la ciencia reproducible: **Quarto** (`.qmd`), utilizando Posit Cloud.

**Pasos a seguir:**
1. Abre tu proyecto en Posit Cloud. En lugar de RMarkdown, crea un nuevo **Quarto Document**. Notarás que la estructura es muy similar, pero Quarto es más avanzado para compilar código Python y R de manera híbrida si se requiere en el futuro.
2. Utiliza este *prompt* estratégico con tu asistente de IA (ChatGPT, Gemini, etc.):
   > *"Actúa como un analista de datos avanzado en R y experto en Quarto. En Python, utilicé `scikit-learn` (`train_test_split`) para dividir un dataset climático (Predictoras: Temp_Max, Radiacion, Viento. Objetivo: ETo) en 80% entrenamiento y 20% prueba. Luego entrené una regresión lineal con el 80% y calculé el MAE y RMSE sobre el conjunto de prueba (20%). Necesito replicar exactamente este flujo en R. Enséñame a usar el paquete `rsample` de `tidymodels` para hacer el `initial_split()`. Muestra cómo entrenar el modelo en R con la data de entrenamiento, hacer la predicción con `predict(modelo, newdata = test_data)`, y calcular MAE y RMSE usando el paquete `yardstick` (o funciones base). Redacta la explicación para integrarla en un documento Quarto."*
3. Observa la sintaxis robusta de `tidymodels`. La función `initial_split(datos, prop = 0.8)` es el equivalente exacto a `train_test_split` de Python.
4. **Entrega y GitHub:** Compila tu documento Quarto (Render). Cierra tu "Bitácora de IA" evaluando cómo el agente de IA propuso el código. Discute brevemente con tus pares o el docente cómo iniciarías el proceso de subir este documento renderizado a un repositorio de **GitHub** para que tu código sea 100% auditable por otros ingenieros en el mundo.